# Stage 4 – Evaluation and analysis
**Owners: Kunal (evaluation, cost), Samir (per-source, explainability), Anurag (robustness)** · ICT6001 Applied Project

Uses the held-out test set **once** to compare both models with the shared evaluation script, then runs the deployment simulation, per-source analysis, robustness test and LIME explanations.

**Runtime → Change runtime type → T4 GPU.** About 10–15 minutes. Run Stages 1–3 first.

In [ ]:
# Setup: connect Google Drive and point to the project folder
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/phishing_project'

import os, sys, json, time, random, pickle, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sys.path.insert(0, PROJECT_DIR)
warnings.filterwarnings('ignore')

SEED = 42
DATA_DIR  = os.path.join(PROJECT_DIR, 'data')
OUT_DIR   = os.path.join(PROJECT_DIR, 'outputs')
SPLIT_DIR = os.path.join(OUT_DIR, 'splits')
MODEL_DIR = os.path.join(OUT_DIR, 'models')
FIG_DIR   = os.path.join(OUT_DIR, 'figures')
RUN_LOG   = os.path.join(OUT_DIR, 'run_log.csv')
for d in (OUT_DIR, SPLIT_DIR, MODEL_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
sns.set_theme(style='whitegrid')
C_SVM, C_BERT, C_PHISH, C_LEGIT = '#3B6EA5', '#D9822B', '#C8553D', '#7FB77E'

def savefig(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, name), dpi=200, bbox_inches='tight'); plt.show()
    print('Saved figure:', name)
def save_summary(stage, d):
    with open(os.path.join(OUT_DIR, f'summary_{stage}.json'), 'w') as f: json.dump(d, f, indent=1, default=str)
print('Project folder:', PROJECT_DIR)

In [ ]:
!pip -q install lime

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, accuracy_score, recall_score
from src.evaluation import classification_metrics, mcnemar_test, per_email_latency
from src.obfuscation import Obfuscator
from src.preprocessing import clean_text
from src.run_log import log_run

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; MAX_LEN = 256
train = pd.read_csv(os.path.join(SPLIT_DIR, 'train.csv')); test = pd.read_csv(os.path.join(SPLIT_DIR, 'test.csv'))
for d in (train, test): d['clean'] = d['clean'].fillna(''); d['text'] = d['text'].fillna('')
yte = test['label'].values
with open(os.path.join(MODEL_DIR, 'svm.pkl'), 'rb') as f: m = pickle.load(f)
vec, svm = m['vectorizer'], m['svm']
FINAL_DIR = os.path.join(MODEL_DIR, 'distilbert_final')
tok = AutoTokenizer.from_pretrained(FINAL_DIR)
bert = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR).to(DEVICE).eval()

@torch.no_grad()
def bert_proba(texts, bs=64):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(list(texts[i:i + bs]), truncation=True, max_length=MAX_LEN, padding=True, return_tensors='pt').to(DEVICE)
        with torch.autocast(device_type=DEVICE, enabled=(DEVICE == 'cuda')):
            logits = bert(**enc).logits
        out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)
print(f'Test set: {len(test):,} emails ({int(yte.sum()):,} phishing) | device: {DEVICE}')

## 4.1 Test-set results (Kunal)

In [ ]:
svm_pred, svm_score = svm.predict(vec.transform(test['clean'])), svm.decision_function(vec.transform(test['clean']))
bert_prob = bert_proba(list(test['clean'])); bert_pred = (bert_prob >= 0.5).astype(int)
res = pd.DataFrame({'TF-IDF + SVM': classification_metrics(yte, svm_pred, svm_score),
                    'DistilBERT': classification_metrics(yte, bert_pred, bert_prob)})
print(res.round(4)); res.to_csv(os.path.join(OUT_DIR, 'test_metrics.csv'))
mc = mcnemar_test(yte, svm_pred, bert_pred)
print("\nMcNemar's test (A = SVM, B = DistilBERT):", json.dumps(mc, indent=1))
print('Significant at p < .05' if mc['p_value'] < 0.05 else 'Not significant at p < .05')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
for ax, (name, pred, cmap) in zip(axes, [('TF-IDF + SVM', svm_pred, 'Blues'), ('DistilBERT', bert_pred, 'Oranges')]):
    sns.heatmap(confusion_matrix(yte, pred, labels=[0, 1]), annot=True, fmt=',d', cmap=cmap, cbar=False, ax=ax,
                xticklabels=['Legitimate', 'Phishing'], yticklabels=['Legitimate', 'Phishing'])
    ax.set_title(name); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
savefig('fig08_confusion_matrices.png')

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for name, score, col in [('TF-IDF + SVM', svm_score, C_SVM), ('DistilBERT', bert_prob, C_BERT)]:
    fpr, tpr, _ = roc_curve(yte, score); ax.plot(fpr, tpr, color=col, label=f'{name} (AUC = {roc_auc_score(yte, score):.4f})')
ax.plot([0, 1], [0, 1], ls='--', color='grey', lw=1)
ax.set_title('ROC curves on the test set'); ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.legend(loc='lower right')
savefig('fig09_roc_curves.png')

mets = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
ax = res.loc[mets].astype(float).rename(index={'roc_auc': 'ROC-AUC', 'f1': 'F1'}).plot(kind='bar', figsize=(8, 4), color=[C_SVM, C_BERT], rot=0)
ax.set_ylim(max(0, float(res.loc[mets].astype(float).values.min()) - 0.03), 1.005)
ax.set_title('Test-set performance: TF-IDF + SVM vs DistilBERT', pad=12); ax.set_ylabel('Score')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=2, frameon=False)
for cont in ax.containers: ax.bar_label(cont, fmt='%.4f', fontsize=7)
savefig('fig10_metric_comparison.png')

## 4.2 Deployment simulation: latency and cost (Kunal)

In [ ]:
emails = list(test['clean'].sample(min(500, len(test)), random_state=SEED))
svm_lat = per_email_latency(lambda t: svm.predict(vec.transform([t])), emails)
bert_lat = per_email_latency(lambda t: bert_proba([t]), emails)
s2 = json.load(open(os.path.join(OUT_DIR, 'summary_02_svm.json'))); s3 = json.load(open(os.path.join(OUT_DIR, 'summary_03_distilbert.json')))
cost = pd.DataFrame({'TF-IDF + SVM': {'training_minutes': s2['fit_seconds'] / 60, 'median_latency_ms': svm_lat['median_ms'], 'p95_latency_ms': svm_lat['p95_ms'], 'model_size_mb': s2['size_mb']},
                     'DistilBERT': {'training_minutes': s3['train_minutes_this_session'], 'median_latency_ms': bert_lat['median_ms'], 'p95_latency_ms': bert_lat['p95_ms'], 'model_size_mb': s3['size_mb']}})
print(cost.round(3)); print('Latency hardware: SVM on CPU; DistilBERT on', DEVICE)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, (row, title) in zip(axes, [('training_minutes', 'Training time (min)'), ('median_latency_ms', 'Median latency (ms per email)'), ('model_size_mb', 'Model size (MB)')]):
    bars = ax.bar(['SVM', 'DistilBERT'], cost.loc[row], color=[C_SVM, C_BERT]); ax.set_title(title); ax.bar_label(bars, fmt='%.3g', fontsize=8); ax.margins(y=0.15)
savefig('fig11_cost_comparison.png')

## 4.3 Per-source analysis (Samir)

In [ ]:
rows = []
for src_name, g in test.groupby('source'):
    idx, y = g.index.values, yte[g.index.values]
    for name, pred in [('TF-IDF + SVM', svm_pred), ('DistilBERT', bert_pred)]:
        p = pred[idx]
        rows.append({'source': src_name, 'model': name, 'emails': len(idx), 'phishing_share': round(y.mean(), 3),
                     'accuracy': round(accuracy_score(y, p), 4), 'phishing_recall': round(recall_score(y, p), 4) if y.sum() else None,
                     'false_positive_rate': round(((p == 1) & (y == 0)).sum() / (y == 0).sum(), 4) if (y == 0).sum() else None})
per_src = pd.DataFrame(rows); print(per_src.to_string(index=False)); per_src.to_csv(os.path.join(OUT_DIR, 'per_source_results.csv'), index=False)
pv = per_src.pivot(index='source', columns='model', values='accuracy')[['TF-IDF + SVM', 'DistilBERT']]
ax = pv.plot(kind='barh', figsize=(8, 4.2), color=[C_SVM, C_BERT])
ax.set_xlim(max(0, float(pv.values.min()) - 0.03), 1.008); ax.set_title('Test accuracy by source corpus'); ax.set_xlabel('Accuracy'); ax.set_ylabel('')
for cont in ax.containers: ax.bar_label(cont, fmt='%.3f', fontsize=7)
savefig('fig12_accuracy_by_source.png')

## 4.4 Robustness test: obfuscated phishing emails (Anurag)

In [ ]:
obf = Obfuscator(train.loc[train['label'] == 0, 'clean'], seed=SEED)
phish = test[test['label'] == 1].reset_index(drop=True)
rob = {}
for name, fn in obf.variants().items():
    texts = [clean_text(fn(t)) for t in phish['text']]
    rob[name] = {'TF-IDF + SVM': float(svm.predict(vec.transform(texts)).mean()), 'DistilBERT': float((bert_proba(texts) >= 0.5).mean())}
rob = pd.DataFrame(rob).T; print(f'Phishing recall on {len(phish):,} obfuscated phishing emails:'); print(rob.round(4))
rob.to_csv(os.path.join(OUT_DIR, 'robustness_results.csv'))
ax = rob.plot(kind='bar', figsize=(9, 4), color=[C_SVM, C_BERT], rot=0)
ax.set_ylim(0, 1.05); ax.set_ylabel('Phishing recall'); ax.set_title('Robustness: phishing recall on obfuscated test emails', pad=12)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
for cont in ax.containers: ax.bar_label(cont, fmt='%.3f', fontsize=7)
plt.xticks(fontsize=8); savefig('fig13_robustness.png')

## 4.5 Explainability (LIME) and error analysis (Samir)

In [ ]:
from lime.lime_text import LimeTextExplainer
explainer = LimeTextExplainer(class_names=['legitimate', 'phishing'], random_state=SEED)
lime_out = {}
picks = {'correct_phishing': np.where((bert_pred == 1) & (yte == 1))[0], 'correct_legitimate': np.where((bert_pred == 0) & (yte == 0))[0],
         'missed_phishing': np.where((bert_pred == 0) & (yte == 1))[0]}
for name, idxs in picks.items():
    if len(idxs) == 0: continue
    i = int(idxs[0]); text = ' '.join(test.loc[i, 'clean'].split()[:150])
    e = explainer.explain_instance(text, lambda t: np.c_[1 - bert_proba(t), bert_proba(t)], num_features=10, num_samples=500)
    lime_out[name] = [(w, round(v, 4)) for w, v in e.as_list()]
    fig = e.as_pyplot_figure(); fig.set_size_inches(7, 3.5); plt.title(f'LIME: {name.replace("_", " ")}')
    savefig(f'fig14_lime_{name}.png')

err = test[['source', 'label']].assign(svm_pred=svm_pred, bert_pred=bert_pred, preview=test['clean'].str[:300])
err[(err.svm_pred != err.label) | (err.bert_pred != err.label)].to_csv(os.path.join(OUT_DIR, 'misclassified_examples.csv'), index=False)
errors = {'svm_missed_by_source': err[(err.svm_pred == 0) & (err.label == 1)]['source'].value_counts().to_dict(),
          'bert_missed_by_source': err[(err.bert_pred == 0) & (err.label == 1)]['source'].value_counts().to_dict(),
          'missed_by_both': int(((err.svm_pred == 0) & (err.bert_pred == 0) & (err.label == 1)).sum())}
print(json.dumps(errors, indent=1))

## 4.6 Results summary – copy everything printed below

In [ ]:
summary = {'test_metrics': res.round(4).to_dict(), 'mcnemar': mc, 'cost': cost.round(3).to_dict(), 'per_source': per_src.to_dict(orient='records'),
           'robustness_recall': rob.round(4).to_dict(orient='index'), 'robustness_n_phishing': len(phish), 'lime': lime_out, 'errors': errors}
save_summary('04_evaluate', summary)
log_run(RUN_LOG, 'evaluate', 'both', {'test_emails': len(test)}, {m: {'f1': round(res.loc['f1', m], 4)} for m in res.columns}, 'Kunal, Samir, Anurag')
full = {s: json.load(open(os.path.join(OUT_DIR, f'summary_{s}.json'))) for s in ['01_preprocess', '02_svm', '03_distilbert', '04_evaluate']}
json.dump(full, open(os.path.join(OUT_DIR, 'results_summary.json'), 'w'), indent=1, default=str)
print(json.dumps(full, indent=1, default=str))